In [ ]:
%pip install tensorrt ultralytics

In [ ]:
import torch
import tensorrt as trt

print("GPU Aktif mi?:", torch.cuda.is_available())
print("CUDA Versiyonu:", torch.version.cuda)
print("TensorRT Versiyonu:", trt.__version__)

GPU Aktif mi?: True
CUDA Versiyonu: 12.8
TensorRT Versiyonu: 11.2.1.2


In [ ]:
import zipfile
import os

zip_path = '/content/baseline_images.zip'
extract_to = '/content/baseline_images'
os.makedirs(extract_to, exist_ok=True)

try:
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Zip dosyası açıldı!")
    print("Açılan klasördeki dosya sayısı:", len(os.listdir(extract_to)))
except Exception as e:
    print(f"Zip açılırken sorun oluştu: {e}")

Zip dosyası açıldı!
Açılan klasördeki dosya sayısı: 2


In [ ]:
import os
import time
import shutil
import numpy as np
import torch
from ultralytics import YOLO

#Dosya yolları ve isimlendirme ayarları
PT_MODEL_PATH = "/content/yolov8_gold_best.pt"
ORIGINAL_ENGINE = "/content/yolov8_gold_best.engine" # Elinizdeki FP16 model

FP16_ENGINE =  "/content/yolov8_gold_best_fp16.engine"
INT8_ENGINE =  "/content/yolov8_gold_best_int8.engine"
YAML_PATH =  "/content/thermal_data.yaml"
IMAGES_DIR =  "/content/baseline_images"

#1) kalibrasyon YAML dosyasını oluşturalım
yaml_content = f"""path: {IMAGES_DIR}
train: .
val: .
names:
  0: human
"""

with open(YAML_PATH, "w") as f:
    f.write(yaml_content)

print(f"Kalibrasyon YAML oluşturuldu: {YAML_PATH}")

# 2)Elimizdeki mevcut FP16  .engine dosyasını korumak için adını değiştiriyoruz.
if os.path.exists(ORIGINAL_ENGINE) and not os.path.exists(FP16_ENGINE):
  os.rename(ORIGINAL_ENGINE, FP16_ENGINE)
  print(f"FP16 .engine dosyası değiştirildi: {FP16_ENGINE}")
elif os.path.exists(FP16_ENGINE):
  print(f"FP16 .engine dosyası hazır: {FP16_ENGINE}")
else:
  print(f"FP16 .engine dosyası bulunamadı: {ORIGINAL_ENGINE}, benchmark sadece INT8 için çalışacak")

# INT8 TensorRT Export ve Kalibrasyonu
print(" TensorRT INT8 Quantization başlatılıyor")

model = YOLO(PT_MODEL_PATH)

try:
    print(" TensorRT INT8 Engine kalibre ediliyor")

    #export işlemi sıfırdan .engine dosyası üretecek
    exported_path = model.export(
        format="engine",
        imgsz=640,
        device=0,
        int8=True,
        data= YAML_PATH
    )

    #yeni oluşan .engine dosyasını INT8 ismiyle güncelleyelim
    if os.path.exists(exported_path):
      os.rename(exported_path, INT8_ENGINE)
      print(f"TensorRT INT8 Engine hazır: {INT8_ENGINE}")

except Exception as e:
    print(f"TensorRT INT8 Engine oluşturma hatası: {e}")


Kalibrasyon YAML oluşturuldu: /content/thermal_data.yaml
FP16 .engine dosyası değiştirildi: /content/yolov8_gold_best_fp16.engine
 TensorRT INT8 Quantization başlatılıyor
 TensorRT INT8 Engine kalibre ediliyor
WARNING ⚠️ 'int8' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/yolov8_gold_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime-gpu', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 558ms
Prepared 4 packages in 6.92s
Installed 4 packag

INFO:modelopt.onnx:Starting quantization process for model: /content/yolov8_gold_best.onnx


2026-08-10 13:44:31,813 - [modelopt][onnx] - INFO - Quantization mode: int8


INFO:modelopt.onnx:Quantization mode: int8


2026-08-10 13:44:31,814 - [modelopt][onnx] - INFO - Preprocessing the model /content/yolov8_gold_best.onnx


...

2026-08-10 13:47:27,011 - [modelopt][onnx] - INFO - Validating quantized model


INFO:modelopt.onnx:Validating quantized model


2026-08-10 13:47:27,041 - [modelopt][onnx] - INFO - Quantization process completed


INFO:modelopt.onnx:Quantization process completed


TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 5, 8400) DataType.FLOAT
TensorRT: building INT8 engine as /content/yolov8_gold_best.engine
TensorRT: export success ✅ 522.1s, saved as '/content/yolov8_gold_best.engine' (47.5 MB)

Export complete (524.6s)
Results saved to /content/yolov8_gold_best.engine
Predict:         yolo predict task=detect model=/content/yolov8_gold_best.engine imgsz=640 
Validate:        yolo val task=detect model=/content/yolov8_gold_best.engine imgsz=640 data=/content/thermal_gold.yaml  
Visualize:       https://netron.app
TensorRT INT8 Engine hazır: /content/yolov8_gold_best_int8.engine


In [ ]:
!pip install --force-reinstall torchvision --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 71.7 MB/s eta 0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.26.0+cu128
    Uninstalling torchvision-0.26.0+cu128:
      Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
import os
import time
import numpy as np
from ultralytics import YOLO

# 1. Diskteki model yolları
FP16_ENGINE = "/content/yolov8_gold_best_fp16.engine"
INT8_ENGINE = "/content/yolov8_gold_best_int8.engine"

# Eğer önceden ismi 'yolov8_gold_best.engine' olarak kaldıysa:
if not os.path.exists(FP16_ENGINE) and os.path.exists("/content/yolov8_gold_best.engine"):
    FP16_ENGINE = "/content/yolov8_gold_best.engine"

# 2. Benchmark fonksiyonu
def benchmark_engine(engine_path, iterations=200):
    if not os.path.exists(engine_path):
        print(f"Dosya bulunamadı: {engine_path}")
        return None, None

    print(f"{os.path.basename(engine_path)} yükleniyor ve test ediliyor...")
    model = YOLO(engine_path, task="detect")
    dummy_input = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)

    # GPU ısınma turu (Warmup)
    for _ in range(20):
        _ = model(dummy_input, verbose=False)

    # zaman ölçümü
    start_time = time.perf_counter()
    for _ in range(iterations):
        _ = model(dummy_input, verbose=False)
    end_time = time.perf_counter()

    total_time = end_time - start_time
    avg_latency = (total_time / iterations) * 1000
    fps = iterations / total_time
    return avg_latency, fps

# 3. Testi başlat
print("TensorRT FP16 vs INT8 Performans Benchmark")

fp16_lat, fp16_fps = benchmark_engine(FP16_ENGINE)
int8_lat, int8_fps = benchmark_engine(INT8_ENGINE)

# 4. Sonuç Raporu
print("Benchmark Sonuçları (Tesla T4 GPU)")


if fp16_lat and fp16_fps:
    print(f"FP16 TensorRT Engine -> Gecikme: {fp16_lat:.2f} ms | Hız: {fp16_fps:.1f} FPS")
if int8_lat and int8_fps:
    print(f"INT8 TensorRT Engine -> Gecikme: {int8_lat:.2f} ms | Hız: {int8_fps:.1f} FPS")

if fp16_lat and int8_lat:
    speedup = fp16_lat / int8_lat
    latency_diff = fp16_lat - int8_lat
    print(f"INT8 Modeli, FP16 Modelinden {speedup:.2f}x daha hızlı!")
    print(f"Kare Başına Gecikme Kazanımı: -{latency_diff:.2f} ms")


TensorRT FP16 vs INT8 Performans Benchmark
yolov8_gold_best_fp16.engine yükleniyor ve test ediliyor...
Loading /content/yolov8_gold_best_fp16.engine for TensorRT inference...
yolov8_gold_best_int8.engine yükleniyor ve test ediliyor...
Loading /content/yolov8_gold_best_int8.engine for TensorRT inference...
Benchmark Sonuçları (Tesla T4 GPU)
FP16 TensorRT Engine -> Gecikme: 4.16 ms | Hız: 240.3 FPS
INT8 TensorRT Engine -> Gecikme: 4.04 ms | Hız: 247.4 FPS
INT8 Modeli, FP16 Modelinden 1.03x daha hızlı!
Kare Başına Gecikme Kazanımı: -0.12 ms
